# Public Reactions to the Men's 100m Final at the Paris 2024 Olympic Games

*Module 6: Telling the Game: NLP for Sports Intelligence — MSc in AI Applied to Sports*

## Introduction

The men's 100m final at the Paris 2024 Olympic Games provides a focused context for examining how online audiences discuss a major sporting event. YouTube comments offer a source of public reactions, including expressions of enthusiasm, criticism, and observations about the race. Analysing this language can help explore how spectators interpret and respond to elite sporting performances.

This project will examine English-language YouTube comments using exactly two natural language processing (NLP) techniques: word frequency analysis of bigrams and trigrams, and sentiment analysis with positive, negative, and neutral categories. Together, these techniques will describe recurring expressions and the sentiment conveyed in the selected comments. The analysis will concern the sampled YouTube discussion; it will not represent all spectators or establish commenters' underlying intentions.

This notebook currently presents the research plan and local API key configuration only. No comments have been collected or analysed, and no results are reported.

## Research Question and Objectives

**Research question:** What recurring bigrams and trigrams and what distribution of positive, negative, and neutral sentiment characterise the selected English-language YouTube comments about the men's 100m final at Paris 2024?

The project will pursue the following objectives:

1. Define and document a sample of English-language comments from YouTube videos directly related to the final.
2. Identify and rank the most frequent bigrams and trigrams in the prepared comment text.
3. Classify comments as positive, negative, or neutral and summarise the distribution of these categories.
4. Interpret the descriptive outputs in relation to the sporting context, acknowledging sampling limitations and challenges such as sarcasm and ambiguous language.

## Proposed Data and Methodology

**Data source.** The proposed dataset will consist of publicly available English-language comments on selected YouTube videos covering the men's 100m final at Paris 2024. Video selection, collection method, collection date, sample size, and inclusion criteria will be documented when data collection is undertaken. The sample may be influenced by the videos selected, comment availability, and the collection period.

**Technique 1: Word Frequency Analysis.** Bigrams (two consecutive words) and trigrams (three consecutive words) will be counted and ranked to identify recurring expressions. Text preparation decisions will be documented before analysis to make their effects on the counts transparent. Frequent expressions will be interpreted in context rather than treated as evidence of audience-wide opinions.

**Technique 2: Sentiment Analysis.** Each included comment will be assigned a positive, negative, or neutral label using a method to be selected and justified during implementation. The analysis will summarise these labels while recognising that mixed sentiment, sarcasm, and sports-specific language may complicate classification.

**Expected visualizations.** Separate horizontal bar charts will display the most frequent bigrams and trigrams. A further bar chart will show the number and percentage of comments in each sentiment category. These are planned outputs, not observed findings.

## Local API Key Configuration

Copy `.env.example` to `.env` in the project directory and replace `your_api_key_here` with your YouTube Data API v3 key. Keep `.env` private; it is excluded from Git. The selected notebook environment must have `python-dotenv` installed. Run the notebook with the project directory as its working directory. The cell below loads the local key and checks that it is configured without displaying it or calling the API.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY or not YOUTUBE_API_KEY.strip() or YOUTUBE_API_KEY.strip() == "your_api_key_here":
    raise ValueError("Configure YOUTUBE_API_KEY in your local .env file before continuing.")

print("YouTube API key is configured.")

## Data Source Validation

Before collecting the dataset, this test will verify that the selected video is accessible through the YouTube Data API v3 and that top-level comments are available. Video metadata and a small sample of up to 10 comments will help assess whether the source is suitable for subsequent collection. Comments are requested in relevance order, so this sample is not representative of the full discussion. Their language is not assumed to be English; language eligibility will be assessed in a later stage. The reported comment count, when available, is a video-level statistic and may include replies, whereas this test retrieves only top-level comments. This step performs no NLP analysis.


In [2]:
import requests

VIDEO_ID = "7Xnr805bm4E"
API_BASE = "https://www.googleapis.com/youtube/v3"


def youtube_get(resource, **params):
    """Return API data or a safe diagnostic without exposing request URLs."""
    try:
        response = requests.get(
            f"{API_BASE}/{resource}",
            params={**params, "key": YOUTUBE_API_KEY},
            timeout=30,
        )
    except requests.RequestException:
        print(f"{resource}: network request failed (connection or timeout).")
        return None

    try:
        data = response.json()
    except ValueError:
        print(f"{resource}: invalid JSON response (HTTP {response.status_code}).")
        return None

    if not response.ok or "error" in data:
        # Use structured reason codes only; raw errors may include credentials.
        reasons = [
            error.get("reason", "unknown")
            for error in data.get("error", {}).get("errors", [])
        ]
        if "commentsDisabled" in reasons:
            print("Comments are disabled for this video.")
        else:
            reason = ", ".join(reasons) or "unspecified API error"
            print(f"{resource}: HTTP {response.status_code}; reason: {reason}")
        return None
    return data


video_data = youtube_get("videos", part="snippet,statistics", id=VIDEO_ID)
if video_data is not None:
    videos = video_data.get("items", [])
    if not videos:
        print("No accessible video was returned for the supplied video ID.")
    else:
        video = videos[0]
        snippet = video.get("snippet", {})
        print("Title:", snippet.get("title", "Unavailable"))
        print("Channel:", snippet.get("channelTitle", "Unavailable"))
        print("Publication date:", snippet.get("publishedAt", "Unavailable"))
        print("Available comment count:", video.get("statistics", {}).get("commentCount", "Unavailable"))

        comment_data = youtube_get(
            "commentThreads",
            part="snippet",
            videoId=VIDEO_ID,
            maxResults=10,
            textFormat="plainText",
            order="relevance",
        )
        if comment_data is not None:
            comments = comment_data.get("items", [])[:10]
            print(f"Top-level comments returned: {len(comments)}")
            if not comments:
                print("No top-level comments were returned.")
            for number, item in enumerate(comments, start=1):
                text = item.get("snippet", {}).get("topLevelComment", {}).get("snippet", {}).get("textDisplay", "")
                preview = text if len(text) <= 150 else text[:147] + "..."
                print(f"{number}. {preview}")


Title: Men's 100m Final | Paris Champions
Channel: Olympic Games
Publication date: 2024-08-12T00:00:02Z
Available comment count: 2546
Top-level comments returned: 10
1. The best of the #Paris2024 athletics action  https://oly.ch/P24Athletics
2. you do a 15min video of a 9 second race and 3 minute videos of 1 hour basketball games.
3. Without Bolt we can finally see how close a 100m final can be
4. Imagine running 9.91 and getting last place...
5. the 100 m is such a bloodsport. the guy coming in last clocks 9.91. just crazy
6. This race produced fastest 4th, 5th, 6th, 7th and 8th places ever in 100m history 🔥🔥⚡⚡🥇
7. This Olympics goes on to show just how great Bolt was. Noah won 100m, but couldn't win 200m and didn't participate in the 4x100m relay. Jacobs who ...
8. Bolt sitting at home laughing
9. last place got 9.92. Crazy
10. Imagine what's like to think you're leading the race until the finish line, then finding out that for MERE 0.004 SECONDS you got silver medal. That...


## Data Collection

Top-level comments will be collected from the validated Olympic Games video, `7Xnr805bm4E`, using the YouTube Data API v3 and the existing API helper. Requests will retrieve pages of up to 100 comments in time order, following `nextPageToken` until 1,000 unique comments have been collected or no further pages are available. Duplicate comment IDs will be excluded. The CSV will retain each comment's ID, video ID, original plain-text content, publication timestamp, and like count; replies will not be collected.

Selection is limited to comments accessible through the API at collection time. No language filtering or text cleaning will be applied at this stage, so the dataset may contain multiple languages. Time ordering and the collection cap favour recently posted comments, while deleted, moderated, or otherwise unavailable comments cannot be included. The sample therefore does not represent all viewers or the complete historical discussion. API failures may result in a partial dataset, which will be identified in the collection output. Rerunning the cell will replace `comments.csv` with the new collection.


In [3]:
import csv
import json
from datetime import datetime, timezone

MAX_COMMENTS = 1000
fields = ["comment_id", "video_id", "text", "published_at", "like_count"]
collected_comments = []
seen_ids = set()
duplicate_count = 0
page_token = None
used_page_tokens = set()
collection_status = "Comment limit reached."
collection_started_at = datetime.now(timezone.utc).isoformat()

while len(collected_comments) < MAX_COMMENTS:
    params = {
        "part": "snippet",
        "videoId": VIDEO_ID,
        "maxResults": 100,
        "order": "time",
        "textFormat": "plainText",
    }
    if page_token:
        params["pageToken"] = page_token

    page = youtube_get("commentThreads", **params)
    if page is None:
        collection_status = "Stopped after an API/network error; any collected data are partial."
        break

    items = page.get("items", [])
    if not items:
        collection_status = "No further comments were returned."
        break

    for item in items:
        comment = item["snippet"]["topLevelComment"]
        comment_id = comment["id"]
        if comment_id in seen_ids:
            duplicate_count += 1
            continue
        seen_ids.add(comment_id)
        snippet = comment["snippet"]
        collected_comments.append({
            "comment_id": comment_id,
            "video_id": VIDEO_ID,
            "text": snippet["textDisplay"],
            "published_at": snippet["publishedAt"],
            "like_count": snippet.get("likeCount", 0),
        })
        if len(collected_comments) == MAX_COMMENTS:
            break

    if len(collected_comments) == MAX_COMMENTS:
        break
    page_token = page.get("nextPageToken")
    if not page_token:
        collection_status = "All available pages were retrieved."
        break
    if page_token in used_page_tokens:
        collection_status = "Stopped because the API repeated a page token; data may be partial."
        break
    used_page_tokens.add(page_token)

print("Collection started (UTC):", collection_started_at)
print("Collection status:", collection_status)
print("Unique top-level comments collected:", len(collected_comments))
print("Duplicate comment IDs excluded:", duplicate_count)
if collected_comments:
    dates = [datetime.fromisoformat(row["published_at"].replace("Z", "+00:00"))
             for row in collected_comments]
    print("Publication date range (UTC):", min(dates).isoformat(), "to", max(dates).isoformat())
    with open("comments.csv", "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(collected_comments)
    print("Saved to comments.csv")
    print("Preview of five records:")
    print(json.dumps(collected_comments[:5], ensure_ascii=False, indent=2))
else:
    print("Publication date range: unavailable (no comments collected).")
    print("No CSV written because no comments were collected.")


Collection started (UTC): 2026-10-30T14:38:59.757272+00:00
Collection status: Comment limit reached.
Unique top-level comments collected: 1000
Duplicate comment IDs excluded: 1
Publication date range (UTC): 2024-08-12T10:27:44+00:00 to 2026-10-24T18:03:08+00:00
Saved to comments.csv
Preview of five records:
[
  {
    "comment_id": "Ugxm-aUys1sQRvOr68l4AaABAg",
    "video_id": "7Xnr805bm4E",
    "text": "The best of the #Paris2024 athletics action  https://oly.ch/P24Athletics",
    "published_at": "2024-08-15T14:54:42Z",
    "like_count": 179
  },
  {
    "comment_id": "UgzpFU_lFIA5RkM9pjJ4AaABAg",
    "video_id": "7Xnr805bm4E",
    "text": "6:45",
    "published_at": "2026-10-24T18:03:08Z",
    "like_count": 0
  },
  {
    "comment_id": "UgzCG0cBsfcv8PJ97l14AaABAg",
    "video_id": "7Xnr805bm4E",
    "text": "3:09 This is the most main character moment in sports history",
    "published_at": "2026-10-06T13:58:19Z",
    "like_count": 0
  },
  {
    "comment_id": "Ugy0LUs8Et8NFWY6NZl4AaA